### load the data

In [1]:
from load_index import load_faq_data

In [2]:
documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1299 documents


In [7]:
docs_tax_guide = [doc for doc in documents if doc['manual'] == 'paye-manual']
print(f"Loaded {len(docs_tax_guide)} documents for PAYE manual")

Loaded 279 documents for PAYE manual


### set the data index into sqlite database

In [4]:
import time
from sqlitesearch import TextSearchIndex

In [ ]:
index = TextSearchIndex(
    text_fields=["title", "text"],
    keyword_fields=["manual"],
    db_path="faq-search.db"
)

print(index)


In [10]:
for doc in docs_tax_guide:
    index.add(doc)
    print(f"""Added: {doc["text"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq-search.db") # Sleep for 100ms to avoid overwhelming the database

Added: Employer name:  ……………………………………………………… Employer address: …………...
Added: There may be circumstances where employers agree to voluntar...
Added: Full Payment Submission (FPS) Employment Payment Summary (EP...
Added: Commonly known as 'PAYE Direct Payments procedures'. A DPNI ...
Added: This subject is presented as follows Employer required to de...
Added: The employer can use some tax codes in more than one way. So...
Added: The P2 is a personalised communication telling the customer ...
Added: Arrivals in the UK One of the deciding factors on the operat...
Added: Application for modified PAYE for tax equalised expatriate e...
Added: You should use code NT only in cases covered in this subject...
Added: The primary intention of K codes is to Collect tax on benefi...
Added: Arrangement made, under regulation 141 of the Income Tax (Pa...
Added: This subject is presented as follows Full Payment Submission...
Added: Pay As You Earn (PAYE) special arrangement for Short Term Bu...
Added:

### Reconnect to the database and query the index

In [11]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["title", "text"],
    keyword_fields=["manual"],
    db_path="faq-search.db"
)

In [12]:
sqlite_index.count()

280

check for the dta in the sqlite database

In [19]:
results = sqlite_index.search("How do i confirm my payee?", num_results=2)
[doc['text'] for doc in results]


['From 19 March 2001 when someone claims Jobseeker’s Allowance (JSA) for a partner, that partner will also be required to claim if One or both of the couple were born after 19 March 1976 and are over 18 And They have no dependent children In these circumstances both members of the couple will have to make a joint JSA claim and each will have to meet JSA conditions. This condition was introduced by the Welfare Reform and Pensions Act 1999 And JSA (Joint Claims) Regulations (SI1978/2000) This legislation gives a couple the rights to choose which claimant will receive the JSA payment. However, if they cannot agree then the Department for Work and Pensions (DWP) will make the decision (on behalf of the Secretary of State). The payment will not be split and the rates of JSA will be the same as for all JSA claimants. The joint claim requirement will apply to all those who meet the conditions above whether they have an existing claim on 19 March or make a new / renewal JSA claim after this da

In [21]:
sqlite_index.search("How are arrears of pay handled for closed tax years?", num_results=2)

[{'id': 'paye70023',
  'manual': 'paye-manual',
  'title': 'PAYE operation: aspects of PAYE operation - employer: payment of arrears of pay for closed tax years',
  'text': 'Arrears of pay are earnings paid after the date that an employee became entitled to receive them and are usually paid as a lump sum. Arrears of pay are earnings just as if they had been paid at the right time. You can find more information on arrears of pay at EIM02530. Arrears of pay may arise because An employer or employee discovers that wages or salary paid in an earlier period were less than what should have been paid under the employee’s contract The employer’s payroll or HR systems make an error Equal pay legislation applies and the employer has to pay the arrears to the employee(s) or a court has ordered this Legally, an employee’s tax liability on a payment of arrears arises in the tax year that the employee was originally entitled to be paid the extra amounts, not in the year that payments are eventually 

Build a RAG pipeline that can be used to search the index and return the top 5 results. The function should take in a question and a course name as parameters. The function should return the top 5 results from the index.

In [30]:
from tax_knowledge_agent import TaxKnowledgeAgent
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()

assistant = TaxKnowledgeAgent(
    index=sqlite_index,
    llm_client=openai_client,
)

In [31]:
answer = assistant.rag("What is PAYE?")
print(answer)

I don't know.
